# NOTEBOOK 03: MACHINE LEARNING MODEL TRAINING & COMPARISON
We train and benchmark 3 classical ML models:
1. **Logistic Regression** (with cost-sensitive class balancing)
2. **Random Forest Classifier** (ensemble of decision trees)
3. **XGBoost Classifier** (extreme gradient boosted trees)


In [1]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

sys.path.append(os.path.join("..", "training_scripts"))
from data_loader import prepare_splits, FEATURE_COLUMNS

data = prepare_splits()
X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']


### 3.1 Training Candidate Models

In [2]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, eval_metric="logloss")
}

val_results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)[:, 1] if hasattr(model, "predict_proba") else y_pred
    
    val_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred),
        "Recall": recall_score(y_val, y_pred),
        "F1-Score": f1_score(y_val, y_pred),
        "ROC-AUC": roc_auc_score(y_val, y_prob)
    })

pd.DataFrame(val_results)


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,0.8873,0.421352,0.872941,0.568365,0.961293
1,Random Forest,0.9712,1.000000,0.661176,0.796034,0.974804
2,XGBoost,0.9714,0.989583,0.670588,0.799439,0.980428


### 3.2 Evaluation on Unseen Test Set with Best Model

In [3]:
best_model = models["XGBoost"]
y_test_pred = best_model.predict(X_test)
y_test_prob = best_model.predict_proba(X_test)[:, 1]

print("Test Performance (XGBoost):")
print(f"Accuracy:  {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_test_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_test_pred):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_test_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_test_prob):.4f}")


Test Performance (XGBoost):
Accuracy:  0.9725
Precision: 0.9848
Recall:    0.6871
F1-Score:  0.8094
ROC-AUC:   0.9797
